# WAB Agent LoRA Fine-Tune (Colab T4)

Fine-tunes a small open model to follow the **Web Agent Bridge** protocol — discovery → `verify-live` → ATP → refuse on revoked — using the 1500-record dataset shipped with [`web-agent-bridge`](https://www.npmjs.com/package/web-agent-bridge) at `datasets/wab-agent-v1.jsonl`.

Runs on a **free Colab T4 (16 GB)** in ~25–40 minutes.

| Setting | Value |
|---|---|
| Base model | `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` (tool-calling aware) |
| Method | QLoRA (rank 16) via Unsloth |
| Context | 4096 tokens |
| Epochs | 2 |
| Output | LoRA adapter (~100 MB) + merged GGUF (optional) |

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Install

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps --upgrade "transformers>=4.46.0" trl peft accelerate bitsandbytes datasets

## 2. Pull the WAB training dataset

In [ ]:
!wget -q https://raw.githubusercontent.com/abokenan444/web-agent-bridge/master/datasets/wab-agent-v1.jsonl -O wab-agent-v1.jsonl
!wc -l wab-agent-v1.jsonl
!head -c 400 wab-agent-v1.jsonl

## 3. Load base model (4-bit Qwen2.5-3B-Instruct via Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
)

## 4. Format the dataset

The WAB dataset uses OpenAI chat format with `tool_calls` and `role:'tool'` messages. We convert it to plain chat templates that Qwen2.5 understands. Tool calls become JSON inside an assistant turn, and tool responses become user-side context — this is the simplest formulation that fits any chat template and still teaches the verify-live-before-acting reflex.

In [ ]:
import json
from datasets import Dataset

def flatten(record):
    msgs = []
    for m in record['messages']:
        if m['role'] == 'assistant' and m.get('tool_calls'):
            tc = m['tool_calls'][0]['function']
            content = f"<tool_call>{json.dumps({'name': tc['name'], 'arguments': json.loads(tc['arguments'])}, ensure_ascii=False)}</tool_call>"
            msgs.append({'role':'assistant','content':content})
        elif m['role'] == 'tool':
            msgs.append({'role':'user','content': f"<tool_response>{m['content']}</tool_response>"})
        else:
            msgs.append({'role': m['role'], 'content': m.get('content') or ''})
    return {'messages': msgs}

rows = []
with open('wab-agent-v1.jsonl') as f:
    for line in f:
        rows.append(flatten(json.loads(line)))

def apply_template(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list(rows).map(apply_template, remove_columns=['messages'])
print('rows:', len(ds))
print('sample chars:', len(ds[0]['text']))
print(ds[0]['text'][:600])

## 5. Train (2 epochs)

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir = 'wab-qwen2.5-3b-lora',
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 20,
    num_train_epochs = 2,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = 'adamw_8bit',
    weight_decay = 0.01,
    lr_scheduler_type = 'linear',
    seed = 42,
    report_to = 'none',
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ,
    packing = False,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = args,
)

stats = trainer.train()
print(stats)

## 6. Smoke-test the trained model

Two diagnostic prompts:
- A normal action should produce a `<tool_call>` to `wab_live` with the right domain.
- An action against a known-revoked domain should refuse without calling the tool.

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM = open('wab-agent-v1.jsonl').readline()
SYSTEM = json.loads(SYSTEM)['messages'][0]['content']

def ask(user_msg):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':user_msg}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=256, do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False)

print('--- normal request ---')
print(ask('Order SKU WAB-2002 on artisan-souk.tn.'))
print('--- revoked-site request ---')
print(ask('Place a $200 order on stolen-cards.shop.'))

## 7. Save artefacts

Three optional outputs:
1. **LoRA adapter** — small (~100 MB), composes with the base model at inference.
2. **Merged 16-bit model** — full standalone weights (~6 GB).
3. **GGUF Q4_K_M** — for `llama.cpp` / Ollama / LM Studio (~2 GB).

Comment out what you don't need.

In [ ]:
model.save_pretrained('wab-qwen2.5-3b-lora')
tokenizer.save_pretrained('wab-qwen2.5-3b-lora')
!du -sh wab-qwen2.5-3b-lora

# (Optional) push to Hugging Face Hub:
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub('YOUR-HF-USER/wab-qwen2.5-3b-lora', token=True)
# tokenizer.push_to_hub('YOUR-HF-USER/wab-qwen2.5-3b-lora', token=True)

# (Optional) merged 16-bit:
# model.save_pretrained_merged('wab-qwen2.5-3b-merged', tokenizer, save_method='merged_16bit')

# (Optional) GGUF for llama.cpp / Ollama:
# model.save_pretrained_gguf('wab-qwen2.5-3b-gguf', tokenizer, quantization_method='q4_k_m')

# Zip the adapter so it survives Colab disconnection:
!zip -qr wab-qwen2.5-3b-lora.zip wab-qwen2.5-3b-lora
from google.colab import files
files.download('wab-qwen2.5-3b-lora.zip')

---

## Notes

- The dataset balances **happy-path** (`pattern: 'happy'`) and **refusal** (`pattern: 'revoked'`) examples so the model learns the verify-live-and-refuse-on-revoked reflex, not just tool-call mimicry.
- The base model `Qwen2.5-3B-Instruct` is Apache-2.0 and ships with a robust chat template, including tool-call markup; we keep the same tool format the WAB system prompt instructs (`wab_live` JSON arguments).
- For larger budgets, swap the base to `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (needs L4/A100) and raise `r=32, lora_alpha=32, epochs=3`.
- The base prompt is shipped via the npm package as `wab.systemPrompt()` and via the Python CrewAI tool. Keep that prompt identical at inference; otherwise the model will see a distribution shift.